In [1]:
import os, glob
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers,models
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import Callback,EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import  classification_report,confusion_matrix

In [2]:
file_path_train = 'archive/train'
file_path_test = 'archive/test'

In [3]:
classes = os.listdir(file_path_train)
classes

['happy',
 '.DS_Store',
 'sad',
 'fear',
 'surprise',
 'neutral',
 'angry',
 'disgust']

In [4]:
filepath = list(glob.glob(file_path_train+'/**/*.*'))
filepath

['archive/train/happy/Training_50449107.jpg',
 'archive/train/happy/Training_70433018.jpg',
 'archive/train/happy/Training_85610005.jpg',
 'archive/train/happy/Training_4460748.jpg',
 'archive/train/happy/Training_6312930.jpg',
 'archive/train/happy/Training_25740534.jpg',
 'archive/train/happy/Training_80076077.jpg',
 'archive/train/happy/Training_431681.jpg',
 'archive/train/happy/Training_76432922.jpg',
 'archive/train/happy/Training_53152280.jpg',
 'archive/train/happy/Training_82526594.jpg',
 'archive/train/happy/Training_77219425.jpg',
 'archive/train/happy/Training_39023213.jpg',
 'archive/train/happy/Training_77132618.jpg',
 'archive/train/happy/Training_48076410.jpg',
 'archive/train/happy/Training_50296064.jpg',
 'archive/train/happy/Training_72681057.jpg',
 'archive/train/happy/Training_54604212.jpg',
 'archive/train/happy/Training_76820039.jpg',
 'archive/train/happy/Training_85112475.jpg',
 'archive/train/happy/Training_2762255.jpg',
 'archive/train/happy/Training_27690434

In [5]:
labels = list(map(lambda x: os.path.split(os.path.split(x)[0])[1], filepath))
labels

['happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',
 'happy',


In [6]:
filepath = pd.Series(filepath,name = 'Filepath').astype(str)
labels = pd.Series(labels,name = 'Labels')
data = pd.concat([filepath,labels],axis = 1)
data = data.sample(frac = 1).reset_index(drop = True) #shuffels the dataset
data.head(5)

,Filepath,Labels
0,archive/train/sad/Training_62351892.jpg,sad
1,archive/train/happy/Training_51022786.jpg,happy
2,archive/train/surprise/Training_22508445.jpg,surprise
3,archive/train/neutral/Training_9943124.jpg,neutral
4,archive/train/sad/Training_26918192.jpg,sad


In [7]:
count = data['Labels'].value_counts()
min_count = count.min()

# Create balanced dataset
balanced_data = []

for label in count.index:
    class_samples = data[data['Labels'] == label]
    if len(class_samples) > min_count:
        balanced_data.append(class_samples.sample(min_count, random_state=42))
    else:
        balanced_data.append(class_samples.sample(min_count, replace=True, random_state=42))

# Concatenate balanced data and shuffle
balanced_data = pd.concat(balanced_data).sample(frac=1, random_state=42).reset_index(drop=True)

# Split into training and test sets
Train, Test = train_test_split(balanced_data, test_size=0.2, random_state=69)

In [8]:
Train.shape

(2441, 2)

In [9]:
# Image data generators for training and testing
train_pre_processing = ImageDataGenerator(
    rescale=1./255,  # Normalize pixel values to [0, 1]
    rotation_range=10,  # Random rotation
    width_shift_range=0.1,  # Random horizontal shift
    height_shift_range=0.1,  # Random vertical shift
    shear_range=0.1,  # Shearing
    zoom_range=0.1,  # Random zoom
    horizontal_flip=True,  # Randomly flip images
    fill_mode='nearest'  # Filling strategy for newly created pixels
)

test_pre_processing = ImageDataGenerator(rescale=1./255)  # Only rescale for test data

# Create generators for training, validation, and test data
train_pre_processed = train_pre_processing.flow_from_dataframe(
    dataframe=Train,
    x_col='Filepath',
    y_col='Labels',
    target_size=(48, 48),
    class_mode='categorical',
    color_mode='grayscale',
    batch_size=32,
    shuffle=True,
    seed=42
)

Validation_pre_processed = test_pre_processing.flow_from_dataframe(
    dataframe=Test,
    x_col='Filepath',
    y_col='Labels',
    target_size=(48, 48),
    class_mode='categorical',
    batch_size=32,
    color_mode='grayscale',
    shuffle=False,
    seed=42
)

Test_pre_processed = test_pre_processing.flow_from_dataframe(
    dataframe=Test,
    x_col='Filepath',
    y_col='Labels',
    target_size=(48, 48),
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=32,
    shuffle=False
)



Found 2441 validated image filenames belonging to 7 classes.
Found 611 validated image filenames belonging to 7 classes.
Found 611 validated image filenames belonging to 7 classes.


In [10]:
data_augmentation = keras.Sequential([
    
    layers.RandomFlip("horizontal", input_shape=(48, 48, 1)),
    layers.RandomRotation(0.3),
    layers.RandomZoom(0.45),
])

/opt/homebrew/lib/python3.9/site-packages/keras/src/layers/preprocessing/tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [11]:
model = models.Sequential([
    layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu', input_shape=(48,48,1)),
    layers.MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    Dropout(0.25),
    
    layers.Conv2D(filters=128, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    Dropout(0.3),

    layers.Conv2D(filters=256, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    Dropout(0.3),

    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    Dropout(0.5),
    layers.Dense(7, activation='softmax'), 
])


/opt/homebrew/lib/python3.9/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [13]:
my_callbacks  = [EarlyStopping(monitor='val_accuracy',min_delta=0,patience=10,mode='auto')]
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0.00001)


In [14]:
history = model.fit(train_pre_processed, validation_data = Validation_pre_processed, epochs=1000)

Epoch 1/1000
 4/77 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.1126 - loss: 1.9879  

/opt/homebrew/lib/python3.9/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


77/77 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.1276 - loss: 1.9641 - val_accuracy: 0.1473 - val_loss: 1.9460
Epoch 2/1000
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.1444 - loss: 1.9454 - val_accuracy: 0.1440 - val_loss: 1.9463
Epoch 3/1000
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.1410 - loss: 1.9460 - val_accuracy: 0.1408 - val_loss: 1.9464
Epoch 4/1000
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.1463 - loss: 1.9474 - val_accuracy: 0.1293 - val_loss: 1.9464
Epoch 5/1000
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.1412 - loss: 1.9464 - val_accuracy: 0.1293 - val_loss: 1.9469
Epoch 6/1000
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.1467 - loss: 1.9463 - val_accuracy: 0.1293 - val_loss: 1.9466
Epoch 7/1000
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.1396 - loss: 1.9442 - val_accuracy: 0.1440 - val_loss: 1.9464
Epoch 8/1000
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.1355 - loss: 1.9469 - val_accuracy: 0.1457 

In [15]:
model.save('emotion_model_1000.keras')